In [ ]:
"""
This script is for converting the output of cellbender into Scanpy objects

authors: Roy Oelen
"""

In [26]:
# imports
import scipy.io
import numpy as np
import pandas as pd
from glob import glob
import pathlib
import os
import scanpy as sc
import re

In [19]:
def get_scanpy_object(args):
    """read a scanpy object from an h5 file
        
    Parameters
    ----------
    args : dict
        dictionary containing parameters to read file, needs 'lane_location'
    
    """
    
    # location of the lane
    lane_location = args['lane_location']
    # grab the lane
    path = pathlib.PurePath(lane_location)
    lane = path.name
    # specific lane location
    lane_append = ''

    # get the full paths
    matrix_loc = "".join([lane_location, lane_append, 'cellbender_remove_background_output_filtered.h5'])
    barcodes_loc = "".join([lane_location, lane_append, 'cellbender_remove_background_output_cell_barcodes.csv'])
    # load the files
    counts_matrix = sc.read_10x_h5(matrix_loc)
    barcodes = pd.read_csv(barcodes_loc, sep= '\t', header=None)
    
    # add the study
    counts_matrix.obs['study'] = 'wijst_multiome'
    # add the lane as a column
    counts_matrix.obs['lane'] = lane
    # add the barcode from the experiment
    counts_matrix.obs['barcode'] = counts_matrix.obs_names.tolist()
    # now the same, but without '-1'
    counts_matrix.obs['barcode_bare'] = [re.sub('-\d+', '', x) for x in counts_matrix.obs_names.tolist()]
    # and the barcode plus the lane
    counts_matrix.obs['barcode_lane'] = counts_matrix.obs['barcode_bare'] + '_' + counts_matrix.obs['lane']
    return counts_matrix

In [28]:
# this is where the CellBender matrices are
cellbender_folders_loc = '/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/2023_09_12_cellbender-v0.3.0/default-run/'
# this is where we will store the objects
scanpy_objects_loc = '/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scanpy_preprocess_samples/objects/'
# list all of the lanes that are present
lanes = [os.path.basename(x) for x in glob(''.join([cellbender_folders_loc, '*lane*']), recursive = False)]

In [30]:
# now check each lane, and write each result
for lane in lanes:
    # paste the folder we need
    cellbender_folder_loc = ''.join([cellbender_folders_loc, lane, '/'])
    # check if the file we want to read exists
    if os.path.isfile(''.join([cellbender_folder_loc, 'cellbender_remove_background_output_filtered.h5'])):
        # try to create the file
        scanpy_object = get_scanpy_object({'lane_location' : cellbender_folder_loc})
        # figure out where to store it
        scanpy_object_loc = ''.join([scanpy_objects_loc, 'mo_', lane, '.h5ad'])
        # write to file
        scanpy_object.write(scanpy_object_loc)
    else:
        print(''.join(['missing output for lane ', lane, ', at ', cellbender_folder_loc]))    

/home/umcg-roelen/miniconda3/envs/scrublet_env/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/umcg-roelen/miniconda3/envs/scrublet_env/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/umcg-roelen/miniconda3/envs/scrublet_env/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/umcg-roelen/miniconda3/envs/scrublet_env/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/umcg-roelen/miniconda3/envs/scrublet_e